# Corrección del plegado de estrellas aRRd

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

In [2]:
ruta_catalogo = Path("../data/interim/catalogo_maestro.csv")
ruta_folding_original = Path("../data/interim/curvas_plegadas.parquet")

catalogo = pd.read_csv(
    ruta_catalogo,
    low_memory=False
)

catalogo_arrd = catalogo[
    catalogo["subtipo_rrlyrae"] == "aRRd"
].copy()

print("Catálogo encontrado:", ruta_catalogo.exists())
print("Parquet original encontrado:", ruta_folding_original.exists())
print(f"Filas del catálogo: {len(catalogo):,}")
print(f"Observaciones aRRd: {len(catalogo_arrd):,}")
print(
    "Identificadores aRRd únicos:",
    f'{catalogo_arrd["observacion_id"].nunique():,}'
)
print(
    "Estrellas físicas aRRd:",
    f'{catalogo_arrd["star_id_base"].nunique():,}'
)

display(
    catalogo_arrd[
        [
            "observacion_id",
            "star_id_base",
            "proyecto_origen",
            "region_origen",
            "subtipo_rrlyrae",
            "periodo_catalogo",
            "periodo_secundario",
            "epoca_catalogo",
            "n_obs_validas",
            "ruta_curva"
        ]
    ]
)

Catálogo encontrado: True
Parquet original encontrado: True
Filas del catálogo: 104,008
Observaciones aRRd: 22
Identificadores aRRd únicos: 22
Estrellas físicas aRRd: 22


,observacion_id,star_id_base,proyecto_origen,region_origen,subtipo_rrlyrae,periodo_catalogo,periodo_secundario,epoca_catalogo,n_obs_validas,ruta_curva
38525,OGLEIV_OGLE-SMC-RRLYR-1125,OGLE-SMC-RRLYR-1125,OGLE-IV,SMC,aRRd,0.350540,0.475438,6000.21045,522,..\data\raw\curves\OGLE-IV\query_1786853168.49...
42526,OGLEIV_OGLE-SMC-RRLYR-6399,OGLE-SMC-RRLYR-6399,OGLE-IV,SMC,aRRd,0.427430,0.565192,6000.06557,471,..\data\raw\curves\OGLE-IV\query_1786853168.49...
42610,OGLEIV_OGLE-LMC-RRLYR-00013,OGLE-LMC-RRLYR-00013,OGLE-IV,LMC,aRRd,0.363332,0.495475,6000.08764,403,..\data\raw\curves\OGLE-IV\query_1786853816.92...
46098,OGLEIV_OGLE-LMC-RRLYR-03878,OGLE-LMC-RRLYR-03878,OGLE-IV,LMC,aRRd,0.391356,0.532351,6000.12048,674,..\data\raw\curves\OGLE-IV\query_1786853816.92...
46368,OGLEIV_OGLE-LMC-RRLYR-04176,OGLE-LMC-RRLYR-04176,OGLE-IV,LMC,aRRd,0.373427,0.506581,6000.29616,786,..\data\raw\curves\OGLE-IV\query_1786853816.92...
49185,OGLEIV_OGLE-LMC-RRLYR-07276,OGLE-LMC-RRLYR-07276,OGLE-IV,LMC,aRRd,0.361959,0.492677,6000.26945,636,..\data\raw\curves\OGLE-IV\query_1786853816.92...
50515,OGLEIV_OGLE-LMC-RRLYR-08767,OGLE-LMC-RRLYR-08767,OGLE-IV,LMC,aRRd,0.383104,0.522000,6000.10964,448,..\data\raw\curves\OGLE-IV\query_1786853816.92...
50650,OGLEIV_OGLE-LMC-RRLYR-08917,OGLE-LMC-RRLYR-08917,OGLE-IV,LMC,aRRd,0.362271,0.497235,6000.11139,665,..\data\raw\curves\OGLE-IV\query_1786853816.92...
51505,OGLEIV_OGLE-LMC-RRLYR-09866,OGLE-LMC-RRLYR-09866,OGLE-IV,LMC,aRRd,0.364250,0.498467,6000.14343,710,..\data\raw\curves\OGLE-IV\query_1786853816.92...
52343,OGLEIV_OGLE-LMC-RRLYR-10802,OGLE-LMC-RRLYR-10802,OGLE-IV,LMC,aRRd,0.365461,0.499116,6000.09915,725,..\data\raw\curves\OGLE-IV\query_1786853816.92...


In [3]:
registros_fundamentales = []

for archivo in sorted(
    Path("../data/raw/periods").glob("OGLE-IV-*-aRRd.dat")
):
    partes_nombre = archivo.stem.split("-")
    proyecto = "-".join(partes_nombre[:2])
    region = partes_nombre[2]

    with archivo.open("r", encoding="utf-8") as archivo_abierto:
        for numero_linea, linea in enumerate(archivo_abierto, start=1):
            campos = linea.split()

            registros_fundamentales.append({
                "proyecto_origen": proyecto,
                "region_origen": region,
                "star_id": campos[0],
                "periodo_primer_sobretono_oficial": float(campos[3]),
                "epoca_primer_sobretono_oficial": float(campos[5]),
                "periodo_fundamental": float(campos[11]),
                "epoca_fundamental": float(campos[13]),
                "archivo_parametros_fundamentales": archivo.name
            })

parametros_fundamentales = pd.DataFrame(registros_fundamentales)

catalogo_arrd = catalogo_arrd.merge(
    parametros_fundamentales,
    on=[
        "proyecto_origen",
        "region_origen",
        "star_id"
    ],
    how="left",
    validate="one_to_one"
)

coincide_primer_sobretono = np.isclose(
    catalogo_arrd["periodo_catalogo"],
    catalogo_arrd["periodo_primer_sobretono_oficial"],
    rtol=0,
    atol=1e-10
)

coincide_fundamental = np.isclose(
    catalogo_arrd["periodo_secundario"],
    catalogo_arrd["periodo_fundamental"],
    rtol=0,
    atol=1e-10
)

print(
    "Registros aRRd en archivos oficiales:",
    f"{len(parametros_fundamentales):,}"
)
print(
    "Observaciones seleccionadas con parámetros completos:",
    f'{catalogo_arrd["periodo_fundamental"].notna().sum():,}'
)
print(
    "Primer sobretono coincidente:",
    f"{coincide_primer_sobretono.sum():,} de {len(catalogo_arrd):,}"
)
print(
    "Modo fundamental coincidente:",
    f"{coincide_fundamental.sum():,} de {len(catalogo_arrd):,}"
)
print(
    "Épocas fundamentales ausentes:",
    catalogo_arrd["epoca_fundamental"].isna().sum()
)

Registros aRRd en archivos oficiales: 90
Observaciones seleccionadas con parámetros completos: 22
Primer sobretono coincidente: 22 de 22
Modo fundamental coincidente: 22 de 22
Épocas fundamentales ausentes: 0


In [4]:
ruta_arrd_corregidas = Path(
    "../data/interim/curvas_aRRd_modo_fundamental.parquet"
)

archivo_original = pq.ParquetFile(ruta_folding_original)
esquema_folding = archivo_original.schema_arrow

filas_arrd_corregidas = []

for _, estrella in catalogo_arrd.iterrows():

    curva = pd.read_csv(
        Path(estrella["ruta_curva"]),
        sep=r"\s+",
        header=None,
        names=["tiempo", "magnitud", "error"]
    )

    fase = np.mod(
        (
            curva["tiempo"].to_numpy(dtype=np.float64)
            - estrella["epoca_fundamental"]
        )
        / estrella["periodo_fundamental"],
        1.0
    )

    orden = np.argsort(fase)

    fase_ordenada = fase[orden].astype(np.float64)
    magnitud_ordenada = (
        curva["magnitud"].to_numpy(dtype=np.float32)[orden]
    )
    error_ordenado = (
        curva["error"].to_numpy(dtype=np.float32)[orden]
    )

    filas_arrd_corregidas.append({
        "observacion_id": estrella["observacion_id"],
        "fase": fase_ordenada.tolist(),
        "magnitud": magnitud_ordenada.tolist(),
        "error": error_ordenado.tolist(),
        "n_obs_folding": len(fase_ordenada)
    })

tabla_arrd_corregidas = pa.Table.from_pylist(
    filas_arrd_corregidas,
    schema=esquema_folding
)

pq.write_table(
    tabla_arrd_corregidas,
    ruta_arrd_corregidas,
    compression="zstd"
)

print("Archivo creado:", ruta_arrd_corregidas.resolve())
print(f"Curvas corregidas: {tabla_arrd_corregidas.num_rows:,}")
print(
    "Mediciones almacenadas:",
    f'{sum(fila["n_obs_folding"] for fila in filas_arrd_corregidas):,}'
)
print(
    "Identificadores únicos:",
    len({
        fila["observacion_id"]
        for fila in filas_arrd_corregidas
    })
)
print(
    "Fases fuera de [0,1):",
    sum(
        np.sum(
            (np.asarray(fila["fase"]) < 0)
            | (np.asarray(fila["fase"]) >= 1)
        )
        for fila in filas_arrd_corregidas
    )
)

Archivo creado: C:\Users\adanm\Code\tesis\data\interim\curvas_aRRd_modo_fundamental.parquet
Curvas corregidas: 22
Mediciones almacenadas: 12,771
Identificadores únicos: 22
Fases fuera de [0,1): 0


In [5]:
ruta_folding_corregido = Path(
    "../data/interim/curvas_plegadas_corregidas.parquet"
)

correcciones_por_id = {
    fila["observacion_id"]: fila
    for fila in filas_arrd_corregidas
}

ids_corregidos = set(correcciones_por_id)
ids_reemplazados = set()
filas_reemplazadas = 0

archivo_original = pq.ParquetFile(ruta_folding_original)

with pq.ParquetWriter(
    ruta_folding_corregido,
    esquema_folding,
    compression="zstd"
) as escritor:

    for numero_grupo in range(archivo_original.num_row_groups):

        tabla_grupo = archivo_original.read_row_group(numero_grupo)
        ids_grupo = tabla_grupo["observacion_id"].to_pylist()

        ids_presentes = ids_corregidos.intersection(ids_grupo)

        if ids_presentes:
            filas_grupo = tabla_grupo.to_pylist()

            for posicion, fila in enumerate(filas_grupo):
                observacion_id = fila["observacion_id"]

                if observacion_id in correcciones_por_id:
                    filas_grupo[posicion] = correcciones_por_id[
                        observacion_id
                    ]
                    ids_reemplazados.add(observacion_id)
                    filas_reemplazadas += 1

            tabla_grupo = pa.Table.from_pylist(
                filas_grupo,
                schema=esquema_folding
            )

        escritor.write_table(tabla_grupo)

        if (
            (numero_grupo + 1) % 20 == 0
            or numero_grupo + 1 == archivo_original.num_row_groups
        ):
            print(
                f"{numero_grupo + 1} de "
                f"{archivo_original.num_row_groups} "
                "grupos procesados"
            )

print("\nArchivo creado:", ruta_folding_corregido.resolve())
print("Filas reemplazadas:", filas_reemplazadas)
print("Identificadores reemplazados:", len(ids_reemplazados))
print(
    "Correcciones no encontradas:",
    len(ids_corregidos - ids_reemplazados)
)
print(
    "Tamaño:",
    f"{ruta_folding_corregido.stat().st_size / 1024**3:.2f} GB"
)

20 de 105 grupos procesados
40 de 105 grupos procesados
60 de 105 grupos procesados
80 de 105 grupos procesados
100 de 105 grupos procesados
105 de 105 grupos procesados

Archivo creado: C:\Users\adanm\Code\tesis\data\interim\curvas_plegadas_corregidas.parquet
Filas reemplazadas: 22
Identificadores reemplazados: 22
Correcciones no encontradas: 0
Tamaño: 1.16 GB


In [6]:
original = pq.ParquetFile(ruta_folding_original)
corregido = pq.ParquetFile(ruta_folding_corregido)

ids_con_fase_modificada = set()
incidencias_estructura = []
mediciones_corregidas = 0

for numero_grupo in range(original.num_row_groups):

    tabla_original = original.read_row_group(numero_grupo)
    tabla_corregida = corregido.read_row_group(numero_grupo)

    ids_originales = tabla_original["observacion_id"]
    ids_corregidos_grupo = tabla_corregida["observacion_id"]

    if not ids_originales.equals(ids_corregidos_grupo):
        incidencias_estructura.append(
            (numero_grupo, "observacion_id")
        )

    for columna in ["magnitud", "error", "n_obs_folding"]:
        if not tabla_original[columna].equals(
            tabla_corregida[columna]
        ):
            incidencias_estructura.append(
                (numero_grupo, columna)
            )

    fases_originales = tabla_original["fase"].to_pylist()
    fases_corregidas = tabla_corregida["fase"].to_pylist()
    ids_grupo = ids_corregidos_grupo.to_pylist()

    for observacion_id, fase_original, fase_corregida in zip(
        ids_grupo,
        fases_originales,
        fases_corregidas
    ):
        if not np.array_equal(
            np.asarray(fase_original),
            np.asarray(fase_corregida)
        ):
            ids_con_fase_modificada.add(observacion_id)

    mediciones_corregidas += sum(
        tabla_corregida["n_obs_folding"].to_pylist()
    )

print("Filas originales:", f"{original.metadata.num_rows:,}")
print("Filas corregidas:", f"{corregido.metadata.num_rows:,}")
print(
    "Grupos internos originales:",
    original.num_row_groups
)
print(
    "Grupos internos corregidos:",
    corregido.num_row_groups
)
print(
    "Mediciones en el archivo corregido:",
    f"{mediciones_corregidas:,}"
)
print(
    "Fases modificadas:",
    len(ids_con_fase_modificada)
)
print(
    "Fases modificadas exclusivamente en aRRd:",
    ids_con_fase_modificada == ids_corregidos
)
print(
    "aRRd esperadas no modificadas:",
    len(ids_corregidos - ids_con_fase_modificada)
)
print(
    "Incidencias en identificadores, magnitudes, errores o n_obs:",
    len(incidencias_estructura)
)

Filas originales: 104,008
Filas corregidas: 104,008
Grupos internos originales: 105
Grupos internos corregidos: 105
Mediciones en el archivo corregido: 132,232,817
Fases modificadas: 22
Fases modificadas exclusivamente en aRRd: True
aRRd esperadas no modificadas: 0
Incidencias en identificadores, magnitudes, errores o n_obs: 34


In [7]:
original = pq.ParquetFile(ruta_folding_original)
corregido = pq.ParquetFile(ruta_folding_corregido)

filas_no_arrd_modificadas = []
arrd_distintas_de_correccion = []
arrd_con_datos_fotometricos_distintos = []
arrd_validadas = set()
ids_desalineados = []

for numero_grupo in range(original.num_row_groups):

    filas_originales = (
        original.read_row_group(numero_grupo).to_pylist()
    )
    filas_corregidas = (
        corregido.read_row_group(numero_grupo).to_pylist()
    )

    for fila_original, fila_corregida in zip(
        filas_originales,
        filas_corregidas
    ):
        observacion_id = fila_original["observacion_id"]

        if observacion_id != fila_corregida["observacion_id"]:
            ids_desalineados.append(observacion_id)
            continue

        if observacion_id not in ids_corregidos:

            if fila_original != fila_corregida:
                filas_no_arrd_modificadas.append(observacion_id)

        else:
            correccion_esperada = correcciones_por_id[
                observacion_id
            ]

            if fila_corregida != correccion_esperada:
                arrd_distintas_de_correccion.append(
                    observacion_id
                )

            fotometria_original = sorted(
                zip(
                    fila_original["magnitud"],
                    fila_original["error"]
                )
            )

            fotometria_corregida = sorted(
                zip(
                    fila_corregida["magnitud"],
                    fila_corregida["error"]
                )
            )

            if fotometria_original != fotometria_corregida:
                arrd_con_datos_fotometricos_distintos.append(
                    observacion_id
                )

            arrd_validadas.add(observacion_id)

print("Identificadores desalineados:", len(ids_desalineados))
print(
    "Curvas no aRRd modificadas:",
    len(filas_no_arrd_modificadas)
)
print(
    "aRRd distintas del archivo de corrección:",
    len(arrd_distintas_de_correccion)
)
print(
    "aRRd con magnitudes o errores alterados:",
    len(arrd_con_datos_fotometricos_distintos)
)
print("aRRd validadas:", len(arrd_validadas))
print(
    "Se validaron exactamente las 22 aRRd:",
    arrd_validadas == ids_corregidos
)

Identificadores desalineados: 0
Curvas no aRRd modificadas: 0
aRRd distintas del archivo de corrección: 0
aRRd con magnitudes o errores alterados: 0
aRRd validadas: 22
Se validaron exactamente las 22 aRRd: True


In [8]:
ruta_catalogo_corregido = Path(
    "../data/interim/catalogo_maestro_corregido.csv"
)

catalogo_corregido = catalogo.copy()

modo_por_subtipo = {
    "RRab": "modo_fundamental",
    "RRc": "primer_sobretono",
    "RRd": "primer_sobretono",
    "RRe": "segundo_sobretono",
    "aRRd": "modo_fundamental"
}

catalogo_corregido["modo_plegado"] = (
    catalogo_corregido["subtipo_rrlyrae"]
    .map(modo_por_subtipo)
)

catalogo_corregido["periodo_plegado"] = (
    catalogo_corregido["periodo_catalogo"]
)

catalogo_corregido["epoca_plegado"] = (
    catalogo_corregido["epoca_catalogo"]
)

catalogo_corregido["plegado_corregido"] = False

parametros_arrd_por_id = (
    catalogo_arrd
    .set_index("observacion_id")
)

mascara_arrd = (
    catalogo_corregido["observacion_id"]
    .isin(ids_corregidos)
)

catalogo_corregido.loc[
    mascara_arrd,
    "periodo_plegado"
] = (
    catalogo_corregido.loc[
        mascara_arrd,
        "observacion_id"
    ]
    .map(parametros_arrd_por_id["periodo_fundamental"])
)

catalogo_corregido.loc[
    mascara_arrd,
    "epoca_plegado"
] = (
    catalogo_corregido.loc[
        mascara_arrd,
        "observacion_id"
    ]
    .map(parametros_arrd_por_id["epoca_fundamental"])
)

catalogo_corregido.loc[
    mascara_arrd,
    "plegado_corregido"
] = True

catalogo_corregido["archivo_folding"] = str(
    ruta_folding_corregido
)

catalogo_corregido.to_csv(
    ruta_catalogo_corregido,
    index=False
)

print("Catálogo creado:", ruta_catalogo_corregido.resolve())
print(f"Filas: {len(catalogo_corregido):,}")
print(f"Columnas: {catalogo_corregido.shape[1]}")
print(
    "Observaciones corregidas:",
    f'{catalogo_corregido["plegado_corregido"].sum():,}'
)
print(
    "Parámetros de plegado ausentes:",
    catalogo_corregido[
        ["modo_plegado", "periodo_plegado", "epoca_plegado"]
    ].isna().any(axis=1).sum()
)
print(
    "aRRd plegadas con modo fundamental:",
    (
        catalogo_corregido.loc[
            catalogo_corregido["subtipo_rrlyrae"] == "aRRd",
            "modo_plegado"
        ]
        == "modo_fundamental"
    ).sum()
)

Catálogo creado: C:\Users\adanm\Code\tesis\data\interim\catalogo_maestro_corregido.csv
Filas: 104,008
Columnas: 41
Observaciones corregidas: 22
Parámetros de plegado ausentes: 0
aRRd plegadas con modo fundamental: 22


In [9]:
indice_folding = pq.read_table(
    ruta_folding_corregido,
    columns=["observacion_id", "n_obs_folding"]
).to_pandas()

cruce_final = catalogo_corregido[
    [
        "observacion_id",
        "subtipo_rrlyrae",
        "n_obs_folding",
        "modo_plegado",
        "periodo_catalogo",
        "periodo_secundario",
        "periodo_plegado",
        "plegado_corregido"
    ]
].merge(
    indice_folding,
    on="observacion_id",
    how="outer",
    suffixes=("_catalogo", "_parquet"),
    indicator=True,
    validate="one_to_one"
)

mascara_arrd_final = (
    cruce_final["subtipo_rrlyrae"] == "aRRd"
)

periodos_arrd_correctos = np.isclose(
    cruce_final.loc[
        mascara_arrd_final,
        "periodo_plegado"
    ],
    cruce_final.loc[
        mascara_arrd_final,
        "periodo_secundario"
    ],
    rtol=0,
    atol=1e-10
)

mascara_no_arrd = ~mascara_arrd_final

periodos_restantes_correctos = np.isclose(
    cruce_final.loc[
        mascara_no_arrd,
        "periodo_plegado"
    ],
    cruce_final.loc[
        mascara_no_arrd,
        "periodo_catalogo"
    ],
    rtol=0,
    atol=1e-10
)

print("Resultado del cruce:")
print(cruce_final["_merge"].value_counts())

print(
    "\nIdentificadores duplicados en el Parquet:",
    indice_folding["observacion_id"].duplicated().sum()
)
print(
    "Diferencias en n_obs:",
    (
        cruce_final["n_obs_folding_catalogo"]
        != cruce_final["n_obs_folding_parquet"]
    ).sum()
)
print(
    "aRRd con período fundamental correcto:",
    f"{periodos_arrd_correctos.sum()} de "
    f"{mascara_arrd_final.sum()}"
)
print(
    "Curvas restantes con período original:",
    f"{periodos_restantes_correctos.sum():,} de "
    f"{mascara_no_arrd.sum():,}"
)
print(
    "Filas marcadas como corregidas:",
    f'{cruce_final["plegado_corregido"].sum():,}'
)

Resultado del cruce:
_merge
both          104008
left_only          0
right_only         0
Name: count, dtype: int64

Identificadores duplicados en el Parquet: 0
Diferencias en n_obs: 0
aRRd con período fundamental correcto: 22 de 22
Curvas restantes con período original: 103,986 de 103,986
Filas marcadas como corregidas: 22
